# World model losses

> Anti-collapse regularisers of joint-embedding world models.

In [ ]:
#| default_exp wm.loss

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's world models (`wm/`) (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import torch
import torch.nn.functional as F

In [ ]:
#| export
class SIGReg(torch.nn.Module):
    """
    Sketched isotropic Gaussian regulariser (LeJEPA, https://arxiv.org/abs/2511.08544): pushes the
    embeddings towards an isotropic Gaussian along `num_proj` random directions (Epps-Pulley
    statistic), which prevents representation collapse. Input ``(T, B, D)``.
    """
    def __init__(self, knots: int = 17, num_proj: int = 1024):
        super().__init__()
        self.num_proj = num_proj
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt
        window = torch.exp(-t.square() / 2.0)
        self.register_buffer('t', t)
        self.register_buffer('phi', window)
        self.register_buffer('weights', weights * window)

    def forward(self, proj: torch.Tensor) -> torch.Tensor:
        A = torch.randn(proj.size(-1), self.num_proj, device=proj.device, dtype=proj.dtype)
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        return ((err @ self.weights) * proj.size(-2)).mean()


class VCReg(torch.nn.Module):
    "Variance-covariance regulariser (VICReg-style) of embeddings ``(B, T, D)``."
    def __init__(self, eps: float = 1e-4):
        super().__init__()
        self.eps = eps

    def _std_loss(self, z):
        std = (z.transpose(0, 1).var(dim=1) + self.eps).sqrt()
        return F.relu(1 - std).mean(dim=-1)

    def _cov_loss(self, z):
        B, T, D = z.shape
        zt = z.transpose(0, 1)
        cov = torch.einsum('tbi,tbj->tij', zt, zt) / (B - 1)
        diag = torch.einsum('tii->ti', cov).pow(2).sum(dim=-1)
        return (cov.pow(2).sum(dim=(-1, -2)) - diag) / (D ** 2 - D)

    def forward(self, z: torch.Tensor) -> dict[str, torch.Tensor]:
        z = z - z.mean(dim=0, keepdim=True)
        return {'std_loss': self._std_loss(z).mean(), 'std_t_loss': self._std_loss(z.transpose(0, 1)).mean(),
                'cov_loss': self._cov_loss(z).mean(), 'cov_t_loss': self._cov_loss(z.transpose(0, 1)).mean()}

### Tests

In [ ]:
torch.manual_seed(0)
sigreg = SIGReg(num_proj=256)
gaussian, collapsed = torch.randn(4, 512, 16), torch.randn(4, 1, 16).expand(4, 512, 16)
assert sigreg(gaussian) < sigreg(collapsed)
assert VCReg()(torch.randn(64, 4, 8))['std_loss'] < VCReg()(torch.zeros(64, 4, 8))['std_loss']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()